# **Imports and Initialization**

In [ ]:
pip install pyyaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.makedirs("/content/drive/MyDrive/SISSA_Thesis/src", exist_ok=True)

# **Configuration Loader**

In [ ]:
%%writefile /content/drive/MyDrive/SISSA_Thesis/src/config_loader.py

import yaml
from pathlib import Path

def load_config(config_path: str = None) -> dict:
    """
    Loads pipeline_config.yaml and returns it as a Python dict.

    Args:
        config_path: explicit path to the yaml file.
                     If None, looks for it at the default location.
    Returns:
        config: dict with keys 'paths', 'gemini', 'generation', 'dataset', 'logging'
    """

    if config_path is None:
        # Default: config/ sits one level above src/
        config_path = Path(__file__).resolve().parent.parent / "config" / "pipeline_config.yaml"
    else:
        config_path = Path(config_path)

    if not config_path.exists():
        raise FileNotFoundError(f"Config file not found at: {config_path}")

    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    return config

Overwriting /content/drive/MyDrive/SISSA_Thesis/src/config_loader.py


In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/SISSA_Thesis/src")

from config_loader import load_config

config = load_config()
print(config)

{'paths': {'base': '/content/drive/MyDrive/SISSA_Thesis', 'blocks_dir': '/content/drive/MyDrive/SISSA_Thesis/blocks_parsed', 'pairs_dir': '/content/drive/MyDrive/SISSA_Thesis/pairs', 'final_dir': '/content/drive/MyDrive/SISSA_Thesis/final'}, 'gemini': {'model': 'gemini-3.1-flash-lite', 'api_key': 'gemini_api_key', 'max_retries': 4, 'base_backoff_sec': 5, 'max_tokens': 1024, 'temperature': 0.4, 'sleep_between_calls_sec': 8}, 'generation': {'max_pairs_per_block': 2, 'min_output_chars': 80, 'max_total_chars': 3000}, 'dataset': {'train_ratio': 0.9, 'random_seed': 42}, 'logging': {'skipped_log': '/content/drive/MyDrive/SISSA_Thesis/data/pairs/skipped_blocks.log'}}


In [ ]:
%%writefile /content/drive/MyDrive/SISSA_Thesis/src/setup_dirs.py

from pathlib import Path
from config_loader import load_config

def setup_directories(config: dict = None) -> None:
    """
    Creates all pipeline output directories if they don't already exist.
    Safe to re-run — will not overwrite anything.
    """

    if config is None:
        config = load_config()

    dirs = [
        config["paths"]["blocks_dir"],
        config["paths"]["pairs_dir"],
        config["paths"]["final_dir"],
        # Parent of the skipped log file
        str(Path(config["logging"]["skipped_log"]).parent),
    ]

    for d in dirs:
        Path(d).mkdir(parents=True, exist_ok=True)
        print(f"✓ {d}")

if __name__ == "__main__":
    setup_directories()

Overwriting /content/drive/MyDrive/SISSA_Thesis/src/setup_dirs.py


In [ ]:
from setup_dirs import setup_directories
setup_directories()

✓ /content/drive/MyDrive/SISSA_Thesis/blocks_parsed
✓ /content/drive/MyDrive/SISSA_Thesis/pairs
✓ /content/drive/MyDrive/SISSA_Thesis/final
✓ /content/drive/MyDrive/SISSA_Thesis/data/pairs


# **Template Generator**

In [ ]:
os.makedirs("/content/drive/MyDrive/SISSA_Thesis/src/generator", exist_ok=True)

In [ ]:
%%writefile /content/drive/MyDrive/SISSA_Thesis/src/generator/template_generator.py

import random
from typing import Optional
from pathlib import Path
import sys

# Make sure src/ is on the path when running standalone
sys.path.append(str(Path(__file__).resolve().parent.parent))
from config_loader import load_config

# ── Template bank ─────────────────────────────────────────────────────────────
# Each entry is a (instruction_template, input_field) pair.
# {section_title} and {content} are filled at runtime from the block.

TEMPLATES = {
    "equation": [
        (
            "Explain the physical interpretation of the following equation in the context of {section_title}.",
            "raw_latex"
        ),
        (
            "What does each term in the following equation represent, in the context of {section_title}?",
            "raw_latex"
        ),
        (
            "In the context of {section_title}, what is the role of this equation in the argument?",
            "raw_latex"
        ),
    ],
    "caption": [
        (
            "What does the following figure caption describe, in the context of {section_title}?",
            "content"
        ),
        (
            "Summarise what is being illustrated according to this figure caption from the section on {section_title}.",
            "content"
        ),
    ],
    "table": [
        (
            "What does the following table show in the context of {section_title}?",
            "content"
        ),
        (
            "Interpret the entries of the following table from the section on {section_title}.",
            "content"
        ),
        (
            "What physical quantities are being compared or listed in this table from the section on {section_title}?",
            "content"
        ),
    ],
}

# ── Cleaner ───────────────────────────────────────────────────────────────────

def clean_section_title(title: str) -> str:
    """
    Strips LaTeX commands from section titles so they read naturally
    in instruction strings.

    Handles:
      - \\texorpdfstring{$...$}{plain text}  → keeps the plain text fallback
      - \\emph{}, \\textbf{}, \\textit{}     → keeps inner content
      - Leftover braces                       → removed
    """
    import re

    # \\texorpdfstring{latex}{plain} → plain
    title = re.sub(r'\\texorpdfstring\{[^}]*\}\{([^}]*)\}', r'\1', title)

    # \\emph{x}, \\textbf{x}, \\textit{x} → x
    title = re.sub(r'\\(?:emph|textbf|textit|text)\{([^}]*)\}', r'\1', title)

    # Remove leftover braces
    title = title.replace("{", "").replace("}", "")

    return title.strip()

# ── Generator ─────────────────────────────────────────────────────────────────

def generate_template_pairs(block: dict, config: dict = None) -> list[dict]:
    """
    Given a block dict, returns a list of (instruction, input, metadata) pairs
    using templates. Returns an empty list if the block type has no templates.

    Args:
        block:  a single block dict from Stage 1 JSON output
        config: pipeline config dict (loaded if not provided)

    Returns:
        list of training example dicts (without 'output' field —
        template pairs have no model-generated output yet;
        output will be added by the Gemini generator or left as a note)
    """

    if config is None:
        config = load_config()

    block_type = block.get("block_type")

    if block_type not in TEMPLATES:
        return []  # section_header and text go to Gemini

    max_pairs = config["generation"]["max_pairs_per_block"]
    section_title = clean_section_title(block.get("section_title", ""))

    templates = TEMPLATES[block_type]

    # Sample up to max_pairs templates without replacement
    # If fewer templates than max_pairs, just use all of them
    chosen = random.sample(templates, k=min(max_pairs, len(templates)))

    pairs = []
    for instruction_template, input_field in chosen:
        instruction = instruction_template.format(section_title=section_title)
        input_text  = block.get(input_field, "")

        pair = {
            "instruction": instruction,
            "input":       input_text,
            "output":      "",
            "metadata": {
                "block_id":      block["block_id"],
                "block_type":    block_type,
                "chapter":       block["chapter"],
                "section_title": section_title,
                "source":        "template",
                "task_type":     f"{block_type}_interpretation",
            }
        }
        pairs.append(pair)

    return pairs

Overwriting /content/drive/MyDrive/SISSA_Thesis/src/generator/template_generator.py


## Sanity Check

In [ ]:
from generator.template_generator import generate_template_pairs

In [ ]:
equation_block = {
    "block_id": "ch2_s1_1_block_005",
    "chapter": 2,
    "section": "1.1",
    "section_title": "General Yoga of Planar Abelianization",
    "block_type": "equation",
    "content": "\\begin{equation}\n- i\\frac{k}{4\\pi} \\int \\text{tr} \\left( A \\wedge dA \\right)\n- i\\frac{\\ell}{4\\pi} \\int \\text{tr} \\left( A\\right) \\wedge \\text{tr} \\left( dA \\right) + \\text{SUSY completion}\n\\end{equation}",
    "raw_latex": "\\begin{equation}\n- i\\frac{k}{4\\pi} \\int \\text{tr} \\left( A \\wedge dA \\right)\n- i\\frac{\\ell}{4\\pi} \\int \\text{tr} \\left( A\\right) \\wedge \\text{tr} \\left( dA \\right) + \\text{SUSY completion}\n\\end{equation}",
    "preceding_context": "The equations of motion imply that [math] and [math] do not fluctuate, as expected since they are massive in this vacuum. Expanding in the light fields yields & ( _ - X_i) Q^i_ =0\\,, \\\\ & _i=1^F Q_ ^i, Q^ _i =2 [ -_ _ eff +2+N ) _ - _ =1^N _ _CS interactions +2 _i=1^F | _ - X_i | ]\\,. This precisely matches the equations of motion of a [math] gauge theory with",
    "following_context": "[math] massless fundamentals and Chern--Simons levels [math] , upon comparison with the generic form _i=1^F Q_ ^i, Q^ _i =2 [ - _ eff +k_ eff _ + _ eff _ =1^N _ +2 _i=1^F| _ - X_i| ]\\,, where the CS levels are [math] . We now turn to the mirror of the [math] [math] SQCD, shown in the top right panel of Figure . Solving its equations of motion directly is a highly non-trivial task, due both to the large number of equations and to the existence of multiple possible vacua for fixed real mass parameters."
}

caption_block = {"block_id": "ch2_s1_1_4_block_084",
    "chapter": 2,
    "section": "1.1.4",
    "section_title": "Shifting \\texorpdfstring{$\\ell$ via Witten's $SL(2;\\mathbb Z)$ Action}{l}",
    "block_type": "caption",
    "content": "The $\\mathcal{N}=2$ planar mirror dual of $SU(N)_{-\\frac{F}{2}+N}$ SQCD with $F$ fundamental chiral multiplets.",
    "raw_latex": "\\caption{The $\\mathcal{N}=2$ planar mirror dual of $SU(N)_{-\\frac{F}{2}+N}$ SQCD with $F$ fundamental chiral multiplets.}",
    "preceding_context": "Finally, the partition function satisfies the identity Z_FT[U(N)](,, ) = Z_FT[U(N)](-,-, )\\,, which follows from a redefinition of the integration variables. In the remainder of this work, we will treat the global symmetry as [math] rather than strictly imposing the tracelessness constraints [math] . In practice, this corresponds to relaxing these constraints in the [math] partition function and",
    "following_context": "does not affect any of the physical conclusions. *=2 [math] G[U(N)] [math] FT[U(N)] [math] global symmetry. The resulting duality relates a chiral, non-Abelian, linear quiver gauge theory to a planar, Abelian quiver gauge theory."
  }

text_block = {"block_id": "ch2_s1_1_1_1_block_017",
    "chapter": 2,
    "section": "1.1.1.1",
    "section_title": "",
    "block_type": "text",
    "content": "Matching Global Symmetries The global symmetry of the SQCD theory is, up to discrete factors, $SU(F)\\times U(1)_\\eta$ . On the mirror side, we can match the rank of the global symmetry as follows - in the absence of the monopole superpotential there is a $U(1)$ topological symmetry for each $U(1)$ gauge node. The monopole superpotential breaks the topological symmetries associated to the nodes in a column of the quiver to a diagonal $U(1)$ . Therefore there is a $U(1)^{F-1}$ unbroken topological symmetry associated to the $F-1$ columns of the planar quiver. Furthermore, there is a single $U(1)_\\eta$ flavor symmetry which is preserved by the planar superpotential. In total, the rank of the global symmetry is $F$ , matching the SQCD side. The mirror-like duality implies that the $U(1)^F$ symmetry of the quiver theory is enhanced to $SU(F)\\times U(1)$ , as can be seen from the index calculations as well.",
    "raw_latex": "Matching Global Symmetries The global symmetry of the SQCD theory is, up to discrete factors, $SU(F)\\times U(1)_\\eta$ . On the mirror side, we can match the rank of the global symmetry as follows - in the absence of the monopole superpotential there is a $U(1)$ topological symmetry for each $U(1)$ gauge node. The monopole superpotential breaks the topological symmetries associated to the nodes in a column of the quiver to a diagonal $U(1)$ . Therefore there is a $U(1)^{F-1}$ unbroken topological symmetry associated to the $F-1$ columns of the planar quiver. Furthermore, there is a single $U(1)_\\eta$ flavor symmetry which is preserved by the planar superpotential. In total, the rank of the global symmetry is $F$ , matching the SQCD side. The mirror-like duality implies that the $U(1)^F$ symmetry of the quiver theory is enhanced to $SU(F)\\times U(1)$ , as can be seen from the index calculations as well.",
    "preceding_context": "",
    "following_context": ""
    }

table_block = {"block_id": "ch2_s1_1_1_block_013",
    "chapter": 2,
    "section": "1.1.1",
    "section_title": "Global Symmetries and Gauge Invariant Operators",
    "block_type": "table",
    "content": "\\begin{tabular}{||c | c | c | c||} \n \\hline\n N& F & r & Index \\\\\n \\hline\\hline\n 2 & 4 & 1/5 & $1+ \\eta \\,x^{4/5} +\\eta^2 \\,x^{8/5}-16x^2+\\eta^3 \\, x^{12/5} +\\eta^4 \\, x^{16/5} +(\\eta^5+88 )x^4 +O(x^{21/5}) $ \\\\ \n \\hline\n 2 & 5 & 1/5 & $1+\\eta  x+\\left(\\eta ^2-25\\right) x^2+\\eta ^3 x^3+\\left(\\eta ^4+250\\right) x^4+\\left(\\eta ^5-100 \\eta \\right) x^5+O\\left(x^6\\right)$ \\\\\n \\hline\n 2 & 6 & 1/5 & $1+\\eta  x^{6/5}-36 x^2+\\eta ^2 x^{12/5}+\\eta ^3 x^{18/5}+558 x^4+\\eta ^4 x^{24/5}+O\\left(x^{26/5}\\right)$ \\\\\n \\hline\n 3 & 6 & 1/7 & \n $\\begin{array}{c}1+\\eta  x^{6/7}+\\eta ^2 x^{12/7}-36 x^2+\\eta ^3 x^{18/7}+\\eta ^4 x^{24/7}+558 x^4\\\\ +\\eta ^5 x^{30/7}-225 \\eta  x^{34/7}+O\\left(x^{36/7}\\right) \\end{array}$\n \\\\\n \\hline\n\\end{tabular}",
    "raw_latex": "\\begin{tabular}{||c | c | c | c||} \n \\hline\n N& F & r & Index \\\\\n \\hline\\hline\n 2 & 4 & 1/5 & $1+ \\eta \\,x^{4/5} +\\eta^2 \\,x^{8/5}-16x^2+\\eta^3 \\, x^{12/5} +\\eta^4 \\, x^{16/5} +(\\eta^5+88 )x^4 +O(x^{21/5}) $ \\\\ \n \\hline\n 2 & 5 & 1/5 & $1+\\eta  x+\\left(\\eta ^2-25\\right) x^2+\\eta ^3 x^3+\\left(\\eta ^4+250\\right) x^4+\\left(\\eta ^5-100 \\eta \\right) x^5+O\\left(x^6\\right)$ \\\\\n \\hline\n 2 & 6 & 1/5 & $1+\\eta  x^{6/5}-36 x^2+\\eta ^2 x^{12/5}+\\eta ^3 x^{18/5}+558 x^4+\\eta ^4 x^{24/5}+O\\left(x^{26/5}\\right)$ \\\\\n \\hline\n 3 & 6 & 1/7 & \n $\\begin{array}{c}1+\\eta  x^{6/7}+\\eta ^2 x^{12/7}-36 x^2+\\eta ^3 x^{18/7}+\\eta ^4 x^{24/7}+558 x^4\\\\ +\\eta ^5 x^{30/7}-225 \\eta  x^{34/7}+O\\left(x^{36/7}\\right) \\end{array}$\n \\\\\n \\hline\n\\end{tabular}",
    "preceding_context": "Fixing [math] , [math] , and [math] , one finds a solution & ^(a)_ - ^(a)_ +1=- \\,,\\\\ & ^(a)_ - ^(a+1)_ =2\\,, a<F-N\\,,\\\\ & ^(a)_ - ^(a+1)_ =-2\\,, a F-N\\,,\\\\ & ^(N)_1=-2 \\,. This reproduces the vacuum described in . The resulting light degrees of freedom, determine",
    "following_context": "d from , coincide with those obtained from the [math] partition function analysis. Expanding the equations of motion around this vacuum also reproduces the correct self and mixed Chern--Simons interactions of the planar theory in Figure . The planar superpotential arises from the surviving F-term constraints, while the monopole superpotential is not directly accessible from this analysis."}

for block in [equation_block, caption_block, text_block, table_block]:
    pairs = generate_template_pairs(block)
    for p in pairs:
        print(p["instruction"])
        print(p["metadata"])
        print()

What does each term in the following equation represent, in the context of General Yoga of Planar Abelianization?
{'block_id': 'ch2_s1_1_block_005', 'block_type': 'equation', 'chapter': 2, 'section_title': 'General Yoga of Planar Abelianization', 'source': 'template', 'task_type': 'equation_interpretation'}

Explain the physical interpretation of the following equation in the context of General Yoga of Planar Abelianization.
{'block_id': 'ch2_s1_1_block_005', 'block_type': 'equation', 'chapter': 2, 'section_title': 'General Yoga of Planar Abelianization', 'source': 'template', 'task_type': 'equation_interpretation'}

What does the following figure caption describe, in the context of Shifting l?
{'block_id': 'ch2_s1_1_4_block_084', 'block_type': 'caption', 'chapter': 2, 'section_title': 'Shifting l', 'source': 'template', 'task_type': 'caption_interpretation'}

Summarise what is being illustrated according to this figure caption from the section on Shifting l.
{'block_id': 'ch2_s1_1_4_b

# **Gemini Generator**

In [ ]:
!pip install google-genai

from google import genai

In [ ]:
%%writefile /content/drive/MyDrive/SISSA_Thesis/src/generator/gemini_generator.py

import json
import time
import logging
import re
from pathlib import Path
from typing import Optional
import sys

from google import genai
from google.colab import userdata

sys.path.append(str(Path(__file__).resolve().parent.parent))
from config_loader import load_config

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Prompt builders ───────────────────────────────────────────────────────────

def _build_text_prompt(block: dict) -> str:
    section_title = block.get("section_title", "")
    content       = block.get("content", "")
    preceding     = block.get("preceding_context", "").strip()
    following     = block.get("following_context", "").strip()

    MIN_CONTEXT_CHARS = 30
    context_block = ""
    if len(preceding) >= MIN_CONTEXT_CHARS:
        context_block += f"[Preceding context]: {preceding}\n\n"
    if len(following) >= MIN_CONTEXT_CHARS:
        context_block += f"[Following context]: {following}\n\n"

    prompt = f"""You are helping build a fine-tuning dataset for a physics language model specialising in 3d mirror symmetry.

Given the following passage from a PhD thesis (section: "{section_title}"), generate {{max_pairs}} distinct instruction-output pairs that would train a model to reason about this content.

Each pair must:
- Have a specific, physics-grounded instruction (not generic like "explain this")
- Have a thorough output that directly answers the instruction using the content
- Be self-contained: the output should make sense without re-reading the passage
- Use LaTeX for all mathematical expressions

{context_block}[Passage]:
{content}

Respond ONLY with a JSON array. No preamble, no markdown fences. Format:
[
  {{
    "instruction": "...",
    "output": "..."
  }},
  ...
]"""
    return prompt


def _build_equation_prompt(block: dict) -> Optional[str]:
    section_title = block.get("section_title", "")
    raw_latex     = block.get("raw_latex", block.get("content", ""))
    preceding     = block.get("preceding_context", "").strip()
    following     = block.get("following_context", "").strip()

    MIN_CONTEXT_CHARS = 30

    # No usable context — skip Gemini for this block
    if len(preceding) < MIN_CONTEXT_CHARS and len(following) < MIN_CONTEXT_CHARS:
        return None

    prompt = f"""You are helping build a fine-tuning dataset for a physics language model specialising in 3d mirror symmetry.

Below is a passage from a PhD thesis (section: "{section_title}"). An equation appears in the middle of this passage, marked as [EQUATION].

Passage:
{preceding}
[EQUATION]: {raw_latex}
{following}

Generate {{max_pairs}} question-answer pairs where:
- The QUESTION asks about the role or meaning of [EQUATION] within this specific passage
- The ANSWER is grounded strictly in what the passage says — not general physics knowledge
- If the passage only says the equation is a trivial or simple example, the answer should reflect that honestly
- Use LaTeX for all mathematical expressions

Respond ONLY with a JSON array. No preamble, no markdown fences. Format:
[
  {{
    "instruction": "...",
    "output": "..."
  }},
  ...
]"""
    return prompt


def _build_table_prompt(block: dict) -> str:
    section_title = block.get("section_title", "")
    content       = block.get("content", "")
    preceding     = block.get("preceding_context", "").strip()
    following     = block.get("following_context", "").strip()

    MIN_CONTEXT_CHARS = 30
    context_block = ""
    if len(preceding) >= MIN_CONTEXT_CHARS:
        context_block += f"[Preceding context]: {preceding}\n\n"
    if len(following) >= MIN_CONTEXT_CHARS:
        context_block += f"[Following context]: {following}\n\n"

    prompt = f"""You are helping build a fine-tuning dataset for a physics language model specialising in 3d mirror symmetry.

Given the following table from a PhD thesis (section: "{section_title}"), generate {{max_pairs}} instruction-output pairs focused on interpreting the physical content.

Focus on:
- What the rows and columns represent physically
- Patterns or relationships visible in the data
- Connections to dualities, charges, or symmetry groups

{context_block}[Table in LaTeX]:
{content}

Respond ONLY with a JSON array. No preamble, no markdown fences. Format:
[
  {{
    "instruction": "...",
    "output": "..."
  }},
  ...
]"""
    return prompt


def _build_section_prompt(block: dict, children: list[dict]) -> str:
    section_title = block.get("section_title", "")

    MAX_CHILD_CHARS = 800
    child_summaries = []
    for child in children:
        snippet = child.get("content", "")[:MAX_CHILD_CHARS]
        child_summaries.append(f"[{child['block_type']}]: {snippet}")
    children_text = "\n\n".join(child_summaries)

    prompt = f"""You are helping build a fine-tuning dataset for a physics language model specialising in 3d mirror symmetry.

The following is a section from a PhD thesis titled "{section_title}", along with snippets of its content.

Generate {{max_pairs}} instruction-output pairs where each instruction asks the model to summarise or explain the section's main result or argument.

[Section content snippets]:
{children_text}

Respond ONLY with a JSON array. No preamble, no markdown fences. Format:
[
  {{
    "instruction": "...",
    "output": "..."
  }},
  ...
]"""
    return prompt

# ── Response parser ───────────────────────────────────────────────────────────

def _parse_gemini_response(response_text: str, min_output_chars: int) -> list[dict]:
    # Strip markdown fences
    cleaned = re.sub(r"```(?:json)?", "", response_text).strip().strip("`").strip()

    # Fix unescaped backslashes (LaTeX in JSON)
    cleaned = re.sub(r'(?<!\\)\\(?![\\"/bfnrtu\n])', r'\\\\', cleaned)

    # Fix invalid \uXXXX sequences (e.g. \underline, \underbrace)
    cleaned = re.sub(r'\\u(?![0-9a-fA-F]{4})', r'\\\\u', cleaned)

    try:
        pairs = json.loads(cleaned)
    except json.JSONDecodeError as e:
        logger.warning(f"JSON parse failed: {e} | Raw response: {response_text[:200]}")
        return []

    if not isinstance(pairs, list):
        logger.warning("Gemini response was not a JSON array.")
        return []

    valid = []
    for p in pairs:
        if not isinstance(p, dict):
            continue
        if "instruction" not in p or "output" not in p:
            continue
        if len(p["output"]) < min_output_chars:
            logger.warning(f"Output too short ({len(p['output'])} chars), skipping.")
            continue
        valid.append(p)

    return valid

# ── API caller ────────────────────────────────────────────────────────────────

def _call_gemini(prompt: str, config: dict) -> Optional[str]:
    gemini_cfg  = config["gemini"]
    max_retries = gemini_cfg["max_retries"]
    backoff     = gemini_cfg["base_backoff_sec"]
    model_name  = gemini_cfg["model"]
    sleep_sec   = gemini_cfg.get("sleep_between_calls_sec", 4)

    key_name = gemini_cfg["api_key"]
    api_key  = userdata.get(key_name)
    client   = genai.Client(api_key=api_key)

    # Proactive throttle — stay under 15 req/min free tier limit
    time.sleep(sleep_sec)

    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
            )
            return response.text
        except Exception as e:
            error_str = str(e)
            match = re.search(r"Please retry in ([\d.]+)s", error_str)
            if match:
                wait = int(float(match.group(1))) + 5
            else:
                wait = backoff * (2 ** (attempt - 1))
            logger.warning(f"Gemini attempt {attempt}/{max_retries} failed: {error_str[:120]}. Retrying in {wait}s.")
            time.sleep(wait)

    logger.error("All Gemini retries exhausted.")
    return None

# ── Main dispatcher ───────────────────────────────────────────────────────────

def generate_gemini_pairs(
    block: dict,
    config: dict = None,
    children: list[dict] = None
) -> list[dict]:
    if config is None:
        config = load_config()

    block_type    = block.get("block_type")
    max_pairs     = config["generation"]["max_pairs_per_block"]
    min_out_chars = config["generation"]["min_output_chars"]

    def build_prompt(block, children):
        if block_type == "text":
            return _build_text_prompt(block).replace("{max_pairs}", str(max_pairs))
        elif block_type == "equation":
            p = _build_equation_prompt(block)
            return p.replace("{max_pairs}", str(max_pairs)) if p else None
        elif block_type == "table":
            return _build_table_prompt(block).replace("{max_pairs}", str(max_pairs))
        elif block_type == "section_header":
            ch = children or []
            return _build_section_prompt(block, ch).replace("{max_pairs}", str(max_pairs))
        else:
            return None

    prompt = build_prompt(block, children)

    if prompt is None:
        logger.info(f"Skipping Gemini for block {block.get('block_id')} — no prompt generated.")
        return []

    raw_response = _call_gemini(prompt, config)

    if raw_response is None:
        return []

    pairs = _parse_gemini_response(raw_response, min_out_chars)

    from generator.template_generator import clean_section_title
    section_title = clean_section_title(block.get("section_title", ""))

    results = []
    for p in pairs:
        results.append({
            "instruction": p["instruction"],
            "input":       block.get("raw_latex", block.get("content", "")),
            "output":      p["output"],
            "metadata": {
                "block_id":      block["block_id"],
                "block_type":    block_type,
                "chapter":       block["chapter"],
                "section_title": section_title,
                "source":        "gemini",
                "task_type":     f"{block_type}_explanation",
            }
        })

    return results

Writing /content/drive/MyDrive/SISSA_Thesis/src/generator/gemini_generator.py


In [ ]:
import sys
mods_to_remove = [k for k in sys.modules if 'gemini' in k or 'generator' in k or 'stage2' in k]
for mod in mods_to_remove:
    del sys.modules[mod]

sys.path.append("/content/drive/MyDrive/SISSA_Thesis/src")
import generator.gemini_generator as gm
import inspect
print(inspect.getsource(gm._call_gemini))

def _call_gemini(prompt: str, config: dict) -> Optional[str]:
    gemini_cfg  = config["gemini"]
    max_retries = gemini_cfg["max_retries"]
    backoff     = gemini_cfg["base_backoff_sec"]
    model_name  = gemini_cfg["model"]
    sleep_sec   = gemini_cfg.get("sleep_between_calls_sec", 4)

    key_name = gemini_cfg["api_key"]
    api_key  = userdata.get(key_name)
    client   = genai.Client(api_key=api_key)

    # Proactive throttle — stay under 15 req/min free tier limit
    time.sleep(sleep_sec)

    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
            )
            return response.text
        except Exception as e:
            error_str = str(e)
            match = re.search(r"Please retry in ([\d.]+)s", error_str)
            if match:
                wait = int(float(match.group(1))) + 5
            else:
                wait = backoff * (2 ** (

# Sanity Check

In [ ]:
import os
os.listdir('/content/drive/MyDrive/SISSA_Thesis/blocks_parsed')

['ch01_Geometric_Engineering.json',
 'ch02_Unitary_Mirrors_1.json',
 'ch03_Unitary_Mirrors_2.json',
 'ch04_Symplectic.json',
 'ch05_Non-Susy.json',
 'ch06_Conclusion.json',
 'ch07_SQFTs_A.json',
 'ch08_Unitary_Mirrors_2_A.json',
 'all_blocks.json']

In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/SISSA_Thesis/src")

from generator.gemini_generator import generate_gemini_pairs
from config_loader import load_config

config = load_config()

text_block = {"block_id": "ch2_s1_1_1_1_block_017",
    "chapter": 2,
    "section": "1.1.1.1",
    "section_title": "",
    "block_type": "text",
    "content": "Matching Global Symmetries The global symmetry of the SQCD theory is, up to discrete factors, $SU(F)\\times U(1)_\\eta$ . On the mirror side, we can match the rank of the global symmetry as follows - in the absence of the monopole superpotential there is a $U(1)$ topological symmetry for each $U(1)$ gauge node. The monopole superpotential breaks the topological symmetries associated to the nodes in a column of the quiver to a diagonal $U(1)$ . Therefore there is a $U(1)^{F-1}$ unbroken topological symmetry associated to the $F-1$ columns of the planar quiver. Furthermore, there is a single $U(1)_\\eta$ flavor symmetry which is preserved by the planar superpotential. In total, the rank of the global symmetry is $F$ , matching the SQCD side. The mirror-like duality implies that the $U(1)^F$ symmetry of the quiver theory is enhanced to $SU(F)\\times U(1)$ , as can be seen from the index calculations as well.",
    "raw_latex": "Matching Global Symmetries The global symmetry of the SQCD theory is, up to discrete factors, $SU(F)\\times U(1)_\\eta$ . On the mirror side, we can match the rank of the global symmetry as follows - in the absence of the monopole superpotential there is a $U(1)$ topological symmetry for each $U(1)$ gauge node. The monopole superpotential breaks the topological symmetries associated to the nodes in a column of the quiver to a diagonal $U(1)$ . Therefore there is a $U(1)^{F-1}$ unbroken topological symmetry associated to the $F-1$ columns of the planar quiver. Furthermore, there is a single $U(1)_\\eta$ flavor symmetry which is preserved by the planar superpotential. In total, the rank of the global symmetry is $F$ , matching the SQCD side. The mirror-like duality implies that the $U(1)^F$ symmetry of the quiver theory is enhanced to $SU(F)\\times U(1)$ , as can be seen from the index calculations as well.",
    "preceding_context": "",
    "following_context": ""
    }

pairs = generate_gemini_pairs(text_block, config)
for p in pairs:
    print("INSTRUCTION:", p["instruction"])
    print("OUTPUT:", p["output"][:200])
    print()

INSTRUCTION: Explain how the global symmetry rank is matched between the SQCD theory and its mirror quiver theory, specifically accounting for the effect of the monopole superpotential.
OUTPUT: In the SQCD theory, the global symmetry is $SU(F) 	imes U(1)_\eta$, which has a total rank of $F$. On the mirror side, the quiver theory initially possesses a $U(1)$ topological symmetry for every gau

INSTRUCTION: Detail the mechanism by which the $U(1)^F$ symmetry of the mirror quiver theory is enhanced to $SU(F) 	imes U(1)$ as implied by mirror duality.
OUTPUT: In mirror quiver theories, the initial $U(1)^F$ symmetry arises from the combination of $F-1$ topological symmetries associated with the columns of the quiver and the $U(1)_\eta$ flavor symmetry prese



In [ ]:
%%writefile /content/drive/MyDrive/SISSA_Thesis/src/stage2_runner.py

import json
import logging
from pathlib import Path
import sys

sys.path.append(str(Path(__file__).resolve().parent))

from config_loader import load_config
from setup_dirs import setup_directories
from generator.template_generator import generate_template_pairs
from generator.gemini_generator import generate_gemini_pairs

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# ── Block type routing ────────────────────────────────────────────────────────

TEMPLATE_ONLY       = {"caption"}
GEMINI_ONLY         = {"text", "section_header"}
TEMPLATE_AND_GEMINI = {"equation", "table"}

# ── Helpers ───────────────────────────────────────────────────────────────────

def load_blocks(blocks_path: Path) -> list[dict]:
    """Loads a chapter block JSON and returns the list of blocks."""
    with open(blocks_path, "r") as f:
        data = json.load(f)

    if isinstance(data, list):
        return data
    elif isinstance(data, dict) and "blocks" in data:
        return data["blocks"]
    else:
        raise ValueError(f"Unexpected block JSON format in {blocks_path}")


def group_sections(blocks: list[dict]) -> list[tuple[dict, list[dict]]]:
    """
    Groups blocks into (section_header, [children]) tuples.
    Blocks before the first section header are grouped under a None header.
    Used to pass children to Gemini when processing section_header blocks.
    """
    groups = []
    current_header   = None
    current_children = []

    for block in blocks:
        if block["block_type"] == "section_header":
            if current_header is not None or current_children:
                groups.append((current_header, current_children))
            current_header   = block
            current_children = []
        else:
            current_children.append(block)

    if current_header is not None or current_children:
        groups.append((current_header, current_children))

    return groups


def log_skipped(block: dict, reason: str, log_path: Path) -> None:
    """Appends a skipped block entry to the log file."""
    with open(log_path, "a") as f:
        f.write(json.dumps({
            "block_id":   block.get("block_id"),
            "block_type": block.get("block_type"),
            "reason":     reason
        }) + "\n")

# ── Per-block dispatcher ──────────────────────────────────────────────────────

def process_block(
    block: dict,
    config: dict,
    log_path: Path,
    children: list[dict] = None
) -> list[dict]:
    """
    Dispatches a single block to the appropriate generator(s).
    Returns a list of pairs (may be empty if generation fails).
    """
    block_type = block.get("block_type")
    pairs = []

    if block_type in TEMPLATE_ONLY:
        pairs += generate_template_pairs(block, config)

    elif block_type in GEMINI_ONLY:
        pairs += generate_gemini_pairs(block, config, children=children)

    elif block_type in TEMPLATE_AND_GEMINI:
        pairs += generate_template_pairs(block, config)
        pairs += generate_gemini_pairs(block, config)

    else:
        logger.info(f"Skipping unknown block type: {block_type}")
        log_skipped(block, f"unknown block type: {block_type}", log_path)
        return []

    if not pairs:
        log_skipped(block, "no pairs generated", log_path)

    return pairs

# ── Chapter processor ─────────────────────────────────────────────────────────

def process_chapter(
    blocks_path: Path,
    config: dict,
    log_path: Path
) -> list[dict]:
    """
    Processes all blocks in a single chapter file.
    Returns a flat list of all generated pairs.
    """
    logger.info(f"Processing: {blocks_path.name}")
    blocks = load_blocks(blocks_path)
    groups = group_sections(blocks)

    all_pairs    = []
    total_blocks = len(blocks)
    processed    = 0

    for header, children in groups:

        if header is not None:
            pairs = process_block(header, config, log_path, children=children)
            all_pairs += pairs
            processed += 1

        for block in children:
            pairs = process_block(block, config, log_path)
            all_pairs += pairs
            processed += 1

            if processed % 10 == 0:
                logger.info(f"  {processed}/{total_blocks} blocks processed, "
                            f"{len(all_pairs)} pairs so far.")

    logger.info(f"  Done. {len(all_pairs)} pairs from {blocks_path.name}.")
    return all_pairs

# ── Main runner ───────────────────────────────────────────────────────────────

def run_stage2(config: dict = None) -> None:
    """
    Main entry point for Stage 2.
    Processes all chapter block JSONs and writes pairs to data/pairs/.
    """
    if config is None:
        config = load_config()

    setup_directories(config)

    blocks_dir = Path(config["paths"]["blocks_dir"])
    pairs_dir  = Path(config["paths"]["pairs_dir"])
    log_path   = Path(config["logging"]["skipped_log"])

    log_path.write_text("")  # clear previous log

    block_files = sorted(blocks_dir.glob("*.json"))

    if not block_files:
        logger.error(f"No block JSON files found in {blocks_dir}")
        return

    logger.info(f"Found {len(block_files)} chapter file(s) to process.")

    total_pairs = 0

    for blocks_path in block_files:
        pairs = process_chapter(blocks_path, config, log_path)

        if not pairs:
            logger.warning(f"No pairs generated for {blocks_path.name}, skipping write.")
            continue

        out_name = blocks_path.stem + "_pairs.json"
        out_path = pairs_dir / out_name

        with open(out_path, "w") as f:
            json.dump(pairs, f, indent=2, ensure_ascii=False)

        logger.info(f"  Written: {out_path.name} ({len(pairs)} pairs)")
        total_pairs += len(pairs)

    logger.info(f"\nStage 2 complete. Total pairs generated: {total_pairs}")
    logger.info(f"Skipped blocks log: {log_path}")


if __name__ == "__main__":
    run_stage2()

Writing /content/drive/MyDrive/SISSA_Thesis/src/stage2_runner.py


In [ ]:
from importlib import reload
import config_loader
reload(config_loader)

from config_loader import load_config
config = load_config()
print(config["paths"]["blocks_dir"])

/content/drive/MyDrive/SISSA_Thesis/blocks_parsed


In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/SISSA_Thesis/src")

# Reload modules cleanly
import importlib
import stage2_runner
importlib.reload(stage2_runner)

from stage2_runner import process_chapter
from config_loader import load_config
from pathlib import Path

config   = load_config()
log_path = Path(config["logging"]["skipped_log"])
log_path.write_text("")  # clear log

0

In [ ]:
# Point at just one chapter file
test_file = Path(config["paths"]["blocks_dir"]) / "ch07_SQFTs_A.json"  # adjust name to match yours

pairs = process_chapter(test_file, config, log_path)

print(f"\nTotal pairs: {len(pairs)}")
print("\nFirst pair:")
print(pairs[0])


Total pairs: 36

First pair:
{'instruction': 'Summarize the primary thesis presented in this excerpt regarding the study of supersymmetric gauge theories.', 'input': '\\chapter{Higher Dimensional Constructions}', 'output': 'The excerpt posits that the various dualities and structures observed in lower-dimensional supersymmetric gauge theories can be better understood by adopting a higher-dimensional framework. This suggests that complex phenomena in 3D and 4D theories are specific manifestations of more general structures residing in higher dimensions.', 'metadata': {'block_id': 'ch7_s1_block_001', 'block_type': 'section_header', 'chapter': 7, 'section_title': 'Higher Dimensional Constructions', 'source': 'gemini', 'task_type': 'section_header_explanation'}}


In [ ]:
with open("/content/drive/MyDrive/SISSA_Thesis/data/pairs/ch07_review.txt", "w") as f:
    for i, pair in enumerate(pairs):
        f.write(f"─── Pair {i+1} ───────────────────────────────\n")
        f.write(f"BLOCK:       {pair['metadata']['block_id']}\n")
        f.write(f"TYPE:        {pair['metadata']['block_type']}\n")
        f.write(f"INSTRUCTION: {pair['instruction']}\n")
        f.write(f"OUTPUT:      {pair['output']}\n\n")

print("Written to ch07_review.txt")

Written to ch07_review.txt


In [ ]:
for pair in pairs:
    if pair['metadata']['block_id'] == 'ch7_s1_2_2_block_013':
        print(json.dumps(pair, indent=2, ensure_ascii=False))
        break

{
  "instruction": "What does each term in the following equation represent, in the context of ?",
  "input": "\\begin{equation*}\n    \\mathcal{Z}[\\mathcal{W}_4 \\times \\mathcal{C}_2]\n    \\;\\equiv\\;\n    \\mathcal{Z}[\\mathcal{W}_4]\\big|_{\\mathcal{C}_2}\n    \\;=\\;\n    \\mathcal{Z}[\\mathcal{C}_2]\\big|_{\\mathcal{W}_4} \\, .\n\\end{equation*}",
  "output": "",
  "metadata": {
    "block_id": "ch7_s1_2_2_block_013",
    "block_type": "equation",
    "chapter": 7,
    "section_title": "",
    "source": "template",
    "task_type": "equation_interpretation"
  }
}


In [ ]:
from stage2_runner import load_blocks
from pathlib import Path

blocks = load_blocks(Path("/content/drive/MyDrive/SISSA_Thesis/blocks_a/ch07_SQFTs_A.json"))

for block in blocks:
    if block['block_id'] == 'ch7_s1_2_2_block_013':
        print(json.dumps(block, indent=2, ensure_ascii=False))
        break

{
  "block_id": "ch7_s1_2_2_block_013",
  "chapter": 7,
  "section": "1.2.2",
  "section_title": "",
  "block_type": "equation",
  "content": "\\begin{equation*}\n    \\mathcal{Z}[\\mathcal{W}_4 \\times \\mathcal{C}_2]\n    \\;\\equiv\\;\n    \\mathcal{Z}[\\mathcal{W}_4]\\big|_{\\mathcal{C}_2}\n    \\;=\\;\n    \\mathcal{Z}[\\mathcal{C}_2]\\big|_{\\mathcal{W}_4} \\, .\n\\end{equation*}",
  "raw_latex": "\\begin{equation*}\n    \\mathcal{Z}[\\mathcal{W}_4 \\times \\mathcal{C}_2]\n    \\;\\equiv\\;\n    \\mathcal{Z}[\\mathcal{W}_4]\\big|_{\\mathcal{C}_2}\n    \\;=\\;\n    \\mathcal{Z}[\\mathcal{C}_2]\\big|_{\\mathcal{W}_4} \\, .\n\\end{equation*}",
  "preceding_context": "This construction gives rise to a three-dimensional [math] theory, often denoted [math] , whose protected observables capture invariants of [math] . As in the AGT correspondence, the same six-dimensional partition function may be evaluated by exchanging the roles of spacetime and internal geometry. In particular, supers

In [ ]:
test_file = Path(config["paths"]["blocks_dir"]) / "ch06_Conclusion.json"  # adjust name to match yours

pairs = process_chapter(test_file, config, log_path)

print(f"\nTotal pairs: {len(pairs)}")
print("\nFirst pair:")
print(pairs[0])


Total pairs: 26

First pair:
{'instruction': 'Summarize the primary physical interpretation of 3D N=4 mirror symmetry as described in the text.', 'input': '\\chapter{Conclusions and Outlook}', 'output': 'The text posits that 3D N=4 mirror symmetry can be intuitively understood through the framework of Type IIB string theory and Hanany-Witten brane constructions. This physical realization explains the symmetry as an exchange between Higgs and Coulomb branches, which corresponds to the interchange of flavor and topological symmetries.', 'metadata': {'block_id': 'ch6_s1_block_001', 'block_type': 'section_header', 'chapter': 6, 'section_title': 'Conclusions and Outlook', 'source': 'gemini', 'task_type': 'section_header_explanation'}}


In [ ]:
with open("/content/drive/MyDrive/SISSA_Thesis/data/pairs/ch06_review.txt", "w") as f:
    for i, pair in enumerate(pairs):
        f.write(f"─── Pair {i+1} ───────────────────────────────\n")
        f.write(f"BLOCK:       {pair['metadata']['block_id']}\n")
        f.write(f"TYPE:        {pair['metadata']['block_type']}\n")
        f.write(f"INSTRUCTION: {pair['instruction']}\n")
        f.write(f"OUTPUT:      {pair['output']}\n\n")

print("Written to ch06_review.txt")

Written to ch06_review.txt


In [ ]:
test_file = Path(config["paths"]["blocks_dir"]) / "ch08_Unitary_Mirrors_2_A.json"  # adjust name to match yours

pairs = process_chapter(test_file, config, log_path)

print(f"\nTotal pairs: {len(pairs)}")
print("\nFirst pair:")
print(pairs[0])

with open("/content/drive/MyDrive/SISSA_Thesis/data/pairs/ch08_review.txt", "w") as f:
    for i, pair in enumerate(pairs):
        f.write(f"─── Pair {i+1} ───────────────────────────────\n")
        f.write(f"BLOCK:       {pair['metadata']['block_id']}\n")
        f.write(f"TYPE:        {pair['metadata']['block_type']}\n")
        f.write(f"INSTRUCTION: {pair['instruction']}\n")
        f.write(f"OUTPUT:      {pair['output']}\n\n")

print("Written to ch08_review.txt")


Total pairs: 88

First pair:
{'instruction': 'Summarize the primary objective of the appendix described in the provided text.', 'input': '\\chapter{From the basic \\texorpdfstring{$\\mathcal{N}=4$ Abelian confining dualities to the $\\mathcal{N}=2$}{N=4 to N=2} Abelian dualities}', 'output': 'The appendix aims to derive various self-dualities—specifically for $U(1)_0$ with $[2,2]$ flavors, $U(1)_{1/2}$ with $[2,1]$ flavors, and $U(1)_1$ with $[1,1]$ flavors—by using the assumption that the $U(1)_0$ theory with $[1,1]$ flavors undergoes confinement as a starting point.', 'metadata': {'block_id': 'ch8_s1_block_001', 'block_type': 'section_header', 'chapter': 8, 'section_title': 'From the basic \\texorpdfstring$\\mathcalN=4$ Abelian confining dualities to the $\\mathcalN=2$N=4 to N=2 Abelian dualities', 'source': 'gemini', 'task_type': 'section_header_explanation'}}
Written to ch08_review.txt


In [ ]:
test_file = Path(config["paths"]["blocks_dir"]) / "ch01_Geometric_Engineering.json"  # adjust name to match yours

pairs = process_chapter(test_file, config, log_path)

print(f"\nTotal pairs: {len(pairs)}")
print("\nFirst pair:")
print(pairs[0])

with open("/content/drive/MyDrive/SISSA_Thesis/data/pairs/ch01_review.txt", "w") as f:
    for i, pair in enumerate(pairs):
        f.write(f"─── Pair {i+1} ───────────────────────────────\n")
        f.write(f"BLOCK:       {pair['metadata']['block_id']}\n")
        f.write(f"TYPE:        {pair['metadata']['block_type']}\n")
        f.write(f"INSTRUCTION: {pair['instruction']}\n")
        f.write(f"OUTPUT:      {pair['output']}\n\n")

print("Written to ch01_review.txt")


Total pairs: 931

First pair:
{'instruction': 'Summarize the role of supersymmetry and string theory in the study of quantum field theories as described in the text.', 'input': '\\chapter{Geometric Engineering of Three-Dimensional Supersymmetric Gauge Theories}', 'output': 'Supersymmetry is essential for analyzing strongly coupled quantum field theory dynamics because it constrains quantum corrections, enabling the derivation of exact results and the identification of nonperturbative dualities. String theory further advances this by providing a geometric framework where these dualities can be realized, particularly through D-brane configurations that describe the low-energy dynamics of supersymmetric gauge theories.', 'metadata': {'block_id': 'ch1_s1_block_001', 'block_type': 'section_header', 'chapter': 1, 'section_title': 'Geometric Engineering of Three-Dimensional Supersymmetric Gauge Theories', 'source': 'gemini', 'task_type': 'section_header_explanation'}}
Written to ch01_review.

In [ ]:
test_file = Path(config["paths"]["blocks_dir"]) / "ch02_Unitary_Mirrors_1.json"  # adjust name to match yours

pairs = process_chapter(test_file, config, log_path)

print(f"\nTotal pairs: {len(pairs)}")
print("\nFirst pair:")
print(pairs[0])

with open("/content/drive/MyDrive/SISSA_Thesis/data/pairs/ch02_review.txt", "w") as f:
    for i, pair in enumerate(pairs):
        f.write(f"─── Pair {i+1} ───────────────────────────────\n")
        f.write(f"BLOCK:       {pair['metadata']['block_id']}\n")
        f.write(f"TYPE:        {pair['metadata']['block_type']}\n")
        f.write(f"INSTRUCTION: {pair['instruction']}\n")
        f.write(f"OUTPUT:      {pair['output']}\n\n")

print("Written to ch02_review.txt")


Total pairs: 744

First pair:
{'instruction': "What are 'chiral-planar mirror dualities' as described in the text, and how are they derived?", 'input': '\\chapter{Mirror Duals of \\texorpdfstring{$\\mathcal{N}=2$}{3d N=2} Unitary Chern-Simons-Matter Theories}', 'output': 'Chiral-planar mirror dualities are a class of 3D N=2 dualities that relate non-abelian chiral gauge theories featuring Chern-Simons interactions to purely abelian, planar quiver gauge theories. They are derived by applying real mass deformations to the axial symmetry U(1)τ = U(1)H-C of known N=4 mirror pairs, which reduces the supersymmetry and results in the infrared limit corresponding to these dualities.', 'metadata': {'block_id': 'ch2_s1_block_001', 'block_type': 'section_header', 'chapter': 2, 'section_title': 'Mirror Duals of \\texorpdfstring$\\mathcalN=2$3d N=2 Unitary Chern-Simons-Matter Theories', 'source': 'gemini', 'task_type': 'section_header_explanation'}}
Written to ch02_review.txt


In [ ]:
test_file = Path(config["paths"]["blocks_dir"]) / "ch03_Unitary_Mirrors_2.json"  # adjust name to match yours

pairs = process_chapter(test_file, config, log_path)

print(f"\nTotal pairs: {len(pairs)}")
print("\nFirst pair:")
print(pairs[0])

with open("/content/drive/MyDrive/SISSA_Thesis/data/pairs/ch03_review.txt", "w") as f:
    for i, pair in enumerate(pairs):
        f.write(f"─── Pair {i+1} ───────────────────────────────\n")
        f.write(f"BLOCK:       {pair['metadata']['block_id']}\n")
        f.write(f"TYPE:        {pair['metadata']['block_type']}\n")
        f.write(f"INSTRUCTION: {pair['instruction']}\n")
        f.write(f"OUTPUT:      {pair['output']}\n\n")

print("Written to ch03_review.txt")


Total pairs: 362

First pair:
{'instruction': 'Summarize the relationship between N=4 mirror pairs and the N=2 planar Abelian duals described in the text.', 'input': '\\chapter{The Zoology of \\texorpdfstring{$\\mathcal{N}=2$}{N=2} Unitary CS-SQCD\\texorpdfstring{$_3$}{3} and their Planar Abelian Duals}', 'output': "The text explains that N=2 planar Abelian duals are derived from N=4 mirror pairs through axial mass deformations. These duals are initially restricted to an 'N=4 descendant line' in parameter space, where the Chern-Simons level k is strictly related to the flavor number F and gauge rank N by the formula F = 2|k|+2N.", 'metadata': {'block_id': 'ch3_s1_block_001', 'block_type': 'section_header', 'chapter': 3, 'section_title': 'The Zoology of \\texorpdfstring$\\mathcalN=2$N=2 Unitary CS-SQCD3 and their Planar Abelian Duals', 'source': 'gemini', 'task_type': 'section_header_explanation'}}
Written to ch03_review.txt


In [ ]:
test_file = Path(config["paths"]["blocks_dir"]) / "ch04_Symplectic.json"  # adjust name to match yours

pairs = process_chapter(test_file, config, log_path)

print(f"\nTotal pairs: {len(pairs)}")
print("\nFirst pair:")
print(pairs[0])

with open("/content/drive/MyDrive/SISSA_Thesis/data/pairs/ch04_review.txt", "w") as f:
    for i, pair in enumerate(pairs):
        f.write(f"─── Pair {i+1} ───────────────────────────────\n")
        f.write(f"BLOCK:       {pair['metadata']['block_id']}\n")
        f.write(f"TYPE:        {pair['metadata']['block_type']}\n")
        f.write(f"INSTRUCTION: {pair['instruction']}\n")
        f.write(f"OUTPUT:      {pair['output']}\n\n")

print("Written to ch04_review.txt")


Total pairs: 339

First pair:
{'instruction': 'Summarize the primary research objective of the chapter described in the provided text.', 'input': '\\chapter{Mirror Duals of \\texorpdfstring{$\\mathcal{N}=2$}{N=2} Symplectic Chern-Simons-Matter Theories}', 'output': 'The chapter aims to extend the paradigm of planar Abelianization—previously applied to unitary gauge theories—to three-dimensional N=4 symplectic gauge theories. It specifically addresses the challenges of incorporating orthosymplectic quivers into this framework to better describe the infrared physics of these models.', 'metadata': {'block_id': 'ch4_s1_block_001', 'block_type': 'section_header', 'chapter': 4, 'section_title': 'Mirror Duals of \\texorpdfstring$\\mathcalN=2$N=2 Symplectic Chern-Simons-Matter Theories', 'source': 'gemini', 'task_type': 'section_header_explanation'}}
Written to ch04_review.txt


In [ ]:
test_file = Path(config["paths"]["blocks_dir"]) / "ch05_Non-Susy.json"  # adjust name to match yours

pairs = process_chapter(test_file, config, log_path)

print(f"\nTotal pairs: {len(pairs)}")
print("\nFirst pair:")
print(pairs[0])

with open("/content/drive/MyDrive/SISSA_Thesis/data/pairs/ch05_review.txt", "w") as f:
    for i, pair in enumerate(pairs):
        f.write(f"─── Pair {i+1} ───────────────────────────────\n")
        f.write(f"BLOCK:       {pair['metadata']['block_id']}\n")
        f.write(f"TYPE:        {pair['metadata']['block_type']}\n")
        f.write(f"INSTRUCTION: {pair['instruction']}\n")
        f.write(f"OUTPUT:      {pair['output']}\n\n")

print("Written to ch05_review.txt")


Total pairs: 590

First pair:
{'instruction': 'Summarize the primary focus of 3D Chern-Simons gauge theory research as described in the provided text.', 'input': '\\chapter{Non-Supersymmetric Unitary Dualities}', 'output': 'The text describes the study of IR bosonization dualities in 3D Chern-Simons gauge theories, which focus on relating theories with different microscopic degrees of freedom (bosonic vs. fermionic) that exhibit identical long-distance physics. This research has evolved from large-N analyses to finite-rank gauge groups, establishing a framework for understanding strongly coupled 3D dynamics.', 'metadata': {'block_id': 'ch5_s1_block_001', 'block_type': 'section_header', 'chapter': 5, 'section_title': 'Non-Supersymmetric Unitary Dualities', 'source': 'gemini', 'task_type': 'section_header_explanation'}}
Written to ch05_review.txt
